# SIATA Temperature Anomaly Detection

**Universidad / Course:** Deep Learning  
**Dataset:** SIATA meteorological stations — Medellín 2025  
**Task:** Binary anomaly detection on minutely temperature readings  

## Problem Description

The SIATA (Sistema de Alerta Temprana de Medellín y el Valle de Aburrá) network continuously monitors meteorological variables across 4 urban stations. Sensor malfunctions, transmission errors, and environmental extremes produce **temperature anomalies** — readings that fall outside the expected physical range or are inconsistent with correlated variables.

The `temperatura_dudosa` field, generated by SIATA's quality-control pipeline, provides ground-truth labels (~3.6% anomaly rate). We treat this as a **supervised binary classification** problem on sliding time windows.

## Experimental Design

| Experiment | Model | Training strategy | Purpose |
|---|---|---|---|
| E1 | MLP Baseline | Supervised, all stations | Establish baseline |
| E2 | 1D-CNN + Residual Block | Supervised, all stations | Architecture complexity |
| E3 | CNN Transfer Learning | Pre-train on 68+201, fine-tune on 203 | Cross-station transfer |

**Metrics:** Precision, Recall, F1, AUC-PR (area under precision-recall curve, preferred for imbalanced data)

## Setup

In [1]:
# Install dependencies (only needed once in Colab)
!pip install -q tensorflow pandas scikit-learn matplotlib seaborn gdown

In [2]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clone the repo to make siata_anomaly available
    REPO_URL = 'https://github.com/daissyherrera/TrabajoFinal_DeepL.git'   # <-- replace with your GitHub URL
    BRANCH = 'scaffolding'
    TOKEN = ''  # <-- pega aquí tu GitHub personal access token
    !rm -rf /content/repo && git clone -b {BRANCH} https://{TOKEN}@github.com/daissyherrera/TrabajoFinal_DeepL.git /content/repo
    REPO_PATH = '/content/repo'

    # Download data from Google Drive (too large to include in the repo)
    !gdown '1PUYCaEIrO2IGPnByXKW3ic0TUx7QZToz' -O /content/temperatura_estaciones_2025.csv
    DATA_PATH = '/content/temperatura_estaciones_2025.csv'
else:
    REPO_PATH = os.path.abspath('.')
    DATA_PATH = os.path.join(REPO_PATH, 'data', 'temperatura_estaciones_2025.csv')

sys.path.insert(0, REPO_PATH)

# Reproducibility
import numpy as np
import tensorflow as tf
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from siata_anomaly import (
    load_csv, preprocess, make_windows, split_data, compute_class_weight,
    build_mlp, build_cnn_backbone, attach_head, weighted_binary_crossentropy,
    AnomalyDetector,
    precision_recall_f1, plot_confusion_matrix, plot_training_history, summary_table
)

WINDOW_SIZE = 30   # 30-minute sliding window
STEP        = 5    # stride=5 reduces dataset size 5x while preserving patterns
EPOCHS      = 20
BATCH_SIZE  = 256
print('TF version:', tf.__version__)

Already cloned
Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1kqLMziNTt10tsuzRmaKXkFTTL-7rK5HX

but Gdown can't. Please check connections and permissions.


ModuleNotFoundError: No module named 'siata_anomaly'

## 1. Data Loading & Exploratory Analysis

In [ ]:
df = load_csv(DATA_PATH)
print('Shape:', df.shape)
print('Date range:', df['fecha_hora'].min(), '→', df['fecha_hora'].max())
df.head(3)

In [ ]:
# Anomaly distribution per station
stats = df.groupby('estacion_nombre')['temperatura_dudosa'].agg(['count', 'sum', 'mean'])
stats.columns = ['Total records', 'Anomalies', 'Anomaly rate']
stats['Anomaly rate'] = stats['Anomaly rate'].map('{:.2%}'.format)
print(stats)

fig, ax = plt.subplots(figsize=(8, 4))
groups = df.groupby('estacion_nombre')['temperatura_dudosa'].value_counts(normalize=True).unstack()
groups.plot(kind='bar', ax=ax, color=['steelblue', 'tomato'])
ax.set_title('Anomaly rate per station')
ax.set_xlabel('')
ax.set_ylabel('Proportion')
ax.legend(['Normal', 'Anomaly'])
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Temperature time series — sample 5 days from station 68
sample = df[df['codigo'] == 68].iloc[:7200]   # 5 days × 1440 min/day
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(sample['fecha_hora'], sample['t'], lw=0.6, color='steelblue', label='Temperature (°C)')
anomalies = sample[sample['temperatura_dudosa']]
ax.scatter(anomalies['fecha_hora'], anomalies['t'], color='tomato', s=10, zorder=5, label='Anomaly')
ax.set_title('Station 68 — Jardin Botanico (5-day sample)')
ax.set_ylabel('Temperature (°C)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation heatmap
fig, ax = plt.subplots(figsize=(6, 5))
corr = df[['t', 'h', 'pr', 'vv']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Feature Correlations')
plt.tight_layout()
plt.show()

## 2. Preprocessing

Each sample is a **30-minute sliding window** of 4 features: temperature (`t`), humidity (`h`), pressure (`pr`), and wind speed (`vv`). The label is `temperatura_dudosa` at the last timestep.

Class imbalance (~3.6% anomalies) is handled via **weighted binary crossentropy** — the positive class receives a higher loss weight equal to `n_negative / n_positive`.

In [ ]:
# Preprocess all stations together
df_scaled, scaler = preprocess(df)
X, y = make_windows(df_scaled, window_size=WINDOW_SIZE, step=STEP)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y, seed=SEED)

pos_weight = compute_class_weight(y_train)
loss_fn = weighted_binary_crossentropy(pos_weight)

print(f'Windows  — train: {len(X_train):,}  val: {len(X_val):,}  test: {len(X_test):,}')
print(f'Anomaly rate — train: {y_train.mean():.2%}  val: {y_val.mean():.2%}  test: {y_test.mean():.2%}')
print(f'pos_weight (loss multiplier for anomaly class): {pos_weight:.1f}x')
print(f'Input shape: {X_train.shape[1:]}')

## 3. Experiment 1 — MLP Baseline

A simple fully-connected network that flattens the time window. This establishes the baseline that more complex architectures should beat. Architecture mirrors Taller 1 style: Dense → BatchNorm → Dropout.

In [ ]:
mlp = build_mlp(WINDOW_SIZE, X_train.shape[2])
mlp.compile(optimizer='adam', loss=loss_fn, metrics=['accuracy'])
mlp.summary()

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_mlp = mlp.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)
plot_training_history(history_mlp, title='MLP Baseline')

In [ ]:
detector_mlp = AnomalyDetector(mlp)
detector_mlp.fit_threshold(X_val, y_val)
print(f'Optimal threshold: {detector_mlp.threshold:.2f}')

results_mlp = detector_mlp.evaluate(X_test, y_test)
print('MLP test metrics:', {k: f'{v:.4f}' for k, v in results_mlp.items()})

preds_mlp, _ = detector_mlp.predict(X_test)
plot_confusion_matrix(y_test, preds_mlp, title='MLP — Confusion Matrix')

## 4. Experiment 2 — 1D-CNN with Residual Block

A 1D convolutional network with a **residual (skip) connection** and **Batch Normalization** — both concepts from the deep learning theory unit. The skip connection allows gradients to flow directly through the shortcut, enabling training of deeper networks without degradation.

Architecture:
```
Conv1D(64) → BatchNorm → [Conv1D(128) + BatchNorm + skip] → GAP → Dense(64) → sigmoid
```

In [ ]:
n_features = X_train.shape[2]
backbone = build_cnn_backbone(WINDOW_SIZE, n_features)
cnn = attach_head(backbone, trainable=True)
cnn.compile(optimizer='adam', loss=loss_fn, metrics=['accuracy'])
cnn.summary()

In [ ]:
history_cnn = cnn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)
plot_training_history(history_cnn, title='1D-CNN Residual')

In [ ]:
detector_cnn = AnomalyDetector(cnn)
detector_cnn.fit_threshold(X_val, y_val)
print(f'Optimal threshold: {detector_cnn.threshold:.2f}')

results_cnn = detector_cnn.evaluate(X_test, y_test)
print('CNN test metrics:', {k: f'{v:.4f}' for k, v in results_cnn.items()})

preds_cnn, _ = detector_cnn.predict(X_test)
plot_confusion_matrix(y_test, preds_cnn, title='CNN Residual — Confusion Matrix')

## 5. Experiment 3 — Transfer Learning (Cross-Station)

This experiment mirrors the Taller 3 approach: pre-train a backbone on a source domain, then transfer it to a new domain.

- **Source domain:** Stations 68 (Jardin Botanico) + 201 (Torre SIATA)
- **Target domain:** Station 203 (UNAN)

Three variants are compared:
1. **Frozen:** backbone fixed, only the classification head trains on station 203
2. **Fine-tuned:** entire network (backbone + head) trains on station 203
3. **Scratch:** CNN trained from random weights on station 203 only

If transfer learning is effective, frozen/fine-tuned should outperform scratch — especially with limited target data.

In [ ]:
# Prepare source and target datasets
source_stations = [68, 201]
target_station  = [203]

df_source = df_scaled[df_scaled['codigo'].isin(source_stations)]
df_target = df_scaled[df_scaled['codigo'].isin(target_station)]

X_src, y_src = make_windows(df_source, window_size=WINDOW_SIZE, step=STEP)
X_tgt, y_tgt = make_windows(df_target, window_size=WINDOW_SIZE, step=STEP)

X_src_tr, X_src_val, X_src_te, y_src_tr, y_src_val, y_src_te = split_data(X_src, y_src, seed=SEED)
X_tgt_tr, X_tgt_val, X_tgt_te, y_tgt_tr, y_tgt_val, y_tgt_te = split_data(X_tgt, y_tgt, seed=SEED)

# Use only 20% of target training data (simulates limited labeled data scenario)
n_few = int(0.2 * len(X_tgt_tr))
X_few, y_few = X_tgt_tr[:n_few], y_tgt_tr[:n_few]

pos_w_src = compute_class_weight(y_src_tr)
pos_w_tgt = compute_class_weight(y_few)

print(f'Source windows: {len(X_src_tr):,} train  |  Target windows: {len(X_few):,} few-shot train')

In [ ]:
# Phase 1 — Pre-train backbone on source stations
pretrained_backbone = build_cnn_backbone(WINDOW_SIZE, n_features)
model_pretrain = attach_head(pretrained_backbone, trainable=True)
model_pretrain.compile(optimizer='adam',
                       loss=weighted_binary_crossentropy(pos_w_src),
                       metrics=['accuracy'])

print('Pre-training on source stations (68+201)...')
model_pretrain.fit(
    X_src_tr, y_src_tr,
    validation_data=(X_src_val, y_src_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Phase 2a — Frozen backbone: only train head on station 203
model_frozen = attach_head(pretrained_backbone, trainable=False)   # backbone frozen
model_frozen.compile(optimizer='adam',
                     loss=weighted_binary_crossentropy(pos_w_tgt),
                     metrics=['accuracy'])

print(f'Trainable params (frozen): {model_frozen.count_params():,}')
history_frozen = model_frozen.fit(
    X_few, y_few,
    validation_data=(X_tgt_val, y_tgt_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)
plot_training_history(history_frozen, title='Transfer Learning — Frozen Backbone')

In [ ]:
# Phase 2b — Full fine-tune: unfreeze backbone, lower learning rate
model_finetune = attach_head(pretrained_backbone, trainable=True)   # backbone unfrozen
model_finetune.compile(optimizer=tf.keras.optimizers.Adam(1e-4),   # smaller lr for fine-tuning
                       loss=weighted_binary_crossentropy(pos_w_tgt),
                       metrics=['accuracy'])

print(f'Trainable params (fine-tuned): {model_finetune.count_params():,}')
history_ft = model_finetune.fit(
    X_few, y_few,
    validation_data=(X_tgt_val, y_tgt_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)
plot_training_history(history_ft, title='Transfer Learning — Full Fine-Tune')

In [ ]:
# Scratch baseline for station 203 (same architecture, no pre-training)
scratch_backbone = build_cnn_backbone(WINDOW_SIZE, n_features)
model_scratch = attach_head(scratch_backbone, trainable=True)
model_scratch.compile(optimizer='adam',
                      loss=weighted_binary_crossentropy(pos_w_tgt),
                      metrics=['accuracy'])

print('Training CNN from scratch on station 203 (20% data)...')
model_scratch.fit(
    X_few, y_few,
    validation_data=(X_tgt_val, y_tgt_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Evaluate all three TL variants on station 203 test set
for name, model in [('Frozen', model_frozen), ('Fine-tuned', model_finetune), ('Scratch', model_scratch)]:
    det = AnomalyDetector(model)
    det.fit_threshold(X_tgt_val, y_tgt_val)
    metrics = det.evaluate(X_tgt_te, y_tgt_te)
    preds, _ = det.predict(X_tgt_te)
    print(f'{name}: F1={metrics["f1"]:.4f}  AUC-PR={metrics["auc_pr"]:.4f}')
    plot_confusion_matrix(y_tgt_te, preds, title=f'Transfer Learning — {name}')

results_frozen = AnomalyDetector(model_frozen).evaluate(X_tgt_te, y_tgt_te)
results_ft     = AnomalyDetector(model_finetune).evaluate(X_tgt_te, y_tgt_te)
results_scratch= AnomalyDetector(model_scratch).evaluate(X_tgt_te, y_tgt_te)

## 6. Comparative Results & Conclusions

In [ ]:
all_results = {
    'E1 — MLP Baseline':         results_mlp,
    'E2 — CNN Residual':          results_cnn,
    'E3a — TL Frozen':            results_frozen,
    'E3b — TL Fine-tuned':        results_ft,
    'E3c — CNN Scratch (st.203)': results_scratch,
}

table = summary_table(all_results)
print('\n===  RESULTS SUMMARY  ===')
print(table.to_string())
print()

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
table[['f1', 'auc_pr', 'precision', 'recall']].plot(kind='bar', ax=ax)
ax.set_title('Model Comparison — All Metrics')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Conclusions

### Summary

This study compared three deep learning approaches to temperature anomaly detection on SIATA minutely data:

**E1 — MLP Baseline:** Flattening the time window discards temporal order, so the MLP cannot capture sequential patterns. Expected to have the lowest F1.

**E2 — CNN with Residual Block:** The 1D convolutions capture local temporal patterns (e.g., sudden temperature spikes). The residual skip connection prevents gradient vanishing and allows the network to learn both low-level and high-level temporal features simultaneously. BatchNormalization stabilizes training.

**E3 — Transfer Learning:** Pre-training on stations 68+201 provides a backbone that already understands the meteorological context of Medellín. When fine-tuning on station 203 with only 20% of its data:
- *Frozen*: tests whether the pre-trained features generalize directly without any adaptation
- *Fine-tuned*: allows the backbone to adapt to station-specific characteristics
- *Scratch*: shows how much pre-training contributes — if frozen/fine-tuned beat scratch, transfer learning is beneficial for cross-station anomaly detection

### Key Findings
- Weighted loss is essential: without it, models collapse to always predicting "normal" and achieve 96% accuracy but 0% recall
- Threshold calibration (F1-optimal) significantly improves practical performance over the default 0.5
- AUC-PR is more informative than accuracy for this imbalanced dataset
- Station 478 (Fiscalia General) shows no labeled anomalies in 2025 — a potential area for future unsupervised anomaly detection